DataSets is downloaded from the following resource https://github.com/jmnwong/NSL-KDD-Dataset

### Importing Modules

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [ ]:
pd.set_option("display.max_columns", None)
print(os.listdir())

# Define the column names

In [ ]:
cols = ['duration','protocol_type','service','flag','src_bytes','dst_bytes','land',
'wrong_fragment','urgent','hot','num_failed_logins','logged_in','num_compromised',
'root_shell','su_attempted','num_root','num_file_creations','num_shells',
'num_access_files','num_outbound_cmds','is_host_login','is_guest_login','count',
'srv_count','serror_rate','srv_serror_rate','rerror_rate','srv_rerror_rate',
'same_srv_rate','diff_srv_rate','srv_diff_host_rate','dst_host_count',
'dst_host_srv_count','dst_host_same_srv_rate','dst_host_diff_srv_rate',
'dst_host_same_src_port_rate','dst_host_srv_diff_host_rate','dst_host_serror_rate',
'dst_host_srv_serror_rate','dst_host_rerror_rate','dst_host_srv_rerror_rate',
'label','difficulty']

print(len(cols))

#load both files into pandas


In [ ]:
train = pd.read_csv("KDDTrain+.txt", header=None, names=cols)
test  = pd.read_csv("KDDTest+.txt",  header=None, names=cols)

print("Train shape:", train.shape)
print("Test shape: ", test.shape)

In [ ]:
train.head()

#drop the difficulty column

In [ ]:
train = train.drop(columns=["difficulty"])
test  = test.drop(columns=["difficulty"])

print("Train shape:", train.shape)
print("Test shape: ", test.shape)
print("'difficulty' present :", "difficulty" in train.columns)

#check for missing values and duplicate rows

In [ ]:
print("Missing values (train):", train.isnull().sum().sum())
print("Missing values (test): ", test.isnull().sum().sum())

print("Duplicate rows (train):", train.duplicated().sum())
print("Duplicate rows (test): ", test.duplicated().sum())

print(train.dtypes.value_counts())

#numerical EDA

In [ ]:
train.describe().T

#distribution of the label column

In [ ]:
print(train["label"].value_counts())
print("Distinct labels in train:", train["label"].nunique())

In [ ]:
print("Distinct labels in test: ", test["label"].nunique())
print("Labels only in test:", sorted(set(test["label"]) - set(train["label"])))

#group the attack names

In [ ]:
all_names = sorted(set(train["label"]) | set(test["label"]))
print(len(all_names))
print(all_names)

In [ ]:
normal_names = ['normal']

dos_names = ['back','land','neptune','pod','smurf','teardrop',
             'apache2','udpstorm','processtable','mailbomb']

probe_names = ['satan','ipsweep','nmap','portsweep','mscan','saint']

r2l_names = ['guess_passwd','ftp_write','imap','phf','multihop','warezmaster',
             'warezclient','spy','xlock','xsnoop','snmpguess','snmpgetattack',
             'httptunnel','sendmail','named','worm']

u2r_names = ['buffer_overflow','loadmodule','rootkit','perl','sqlattack','xterm','ps']

In [ ]:
attack_map = {}
for family, names in [('Normal', normal_names), ('DOS', dos_names),
                      ('PROBE', probe_names), ('R2L', r2l_names), ('U2R', u2r_names)]:
    for n in names:
        attack_map[n] = family

missing = [n for n in all_names if n not in attack_map]
print("Names in map:", len(attack_map))
print("Names not placed in any family:", missing)

In [ ]:
train["attack_class"] = train["label"].map(attack_map)
test["attack_class"]  = test["label"].map(attack_map)

print(train["attack_class"].isna().sum(), test["attack_class"].isna().sum())
print(train["attack_class"].value_counts())
print(test["attack_class"].value_counts())

In [ ]:
print((test["label"] == "httptunnel").sum())

In [ ]:
attack_map["httptunnel"] = "U2R"

train["attack_class"] = train["label"].map(attack_map)
test["attack_class"]  = test["label"].map(attack_map)

print(test["attack_class"].value_counts())

In [ ]:
train["is_attack"] = (train["attack_class"] != "Normal").astype(int)
test["is_attack"]  = (test["attack_class"] != "Normal").astype(int)

print(train["is_attack"].value_counts())
print(test["is_attack"].value_counts())

#graphical EDA

In [ ]:
order = ["Normal", "DOS", "PROBE", "R2L", "U2R"]

plt.figure(figsize=(8, 4))
sns.countplot(data=train, x="attack_class", order=order)
plt.title("Number of rows per class (training file)")
plt.xlabel("Class")
plt.ylabel("Number of rows")
plt.show()

In [ ]:

plt.figure(figsize=(8, 4))
sns.countplot(data=test, x="attack_class", order=order)
plt.title("Number of rows per class (test file)")
plt.xlabel("Class")
plt.ylabel("Number of rows")
plt.show()

In [ ]:
print("Train (%):")
print((train["attack_class"].value_counts(normalize=True) * 100).round(1))
print("Test (%):")
print((test["attack_class"].value_counts(normalize=True) * 100).round(1))

In [ ]:
print(train["protocol_type"].value_counts())
print()
print(train["flag"].value_counts())
print()
print("Different services in train:", train["service"].nunique())
print("Different services in test: ", test["service"].nunique())
print("Services only in test:", set(test["service"]) - set(train["service"]))

plt.figure(figsize=(6, 3))
sns.countplot(data=train, x="protocol_type")
plt.title("protocol_type (training file)")
plt.show()

#which protocols do the attacks use

In [ ]:
print(pd.crosstab(train["protocol_type"], train["attack_class"]))
print()
print((pd.crosstab(train["protocol_type"], train["attack_class"], normalize="index") * 100).round(1))

plt.figure(figsize=(8, 4))
sns.countplot(data=train, x="protocol_type", hue="attack_class")
plt.title("Attack classes by protocol (training file)")
plt.show()

In [ ]:
flag_table = pd.crosstab(train["flag"], train["attack_class"])
print(flag_table)
print()

flag_percent = pd.crosstab(train["flag"], train["attack_class"], normalize="index") * 100
print(flag_percent.round(1))

flag_percent.plot(kind="bar", stacked=True, figsize=(9, 4))
plt.title("Class share inside each flag value (training file)")
plt.ylabel("Percent of rows")
plt.show()

#test the service column

In [ ]:
top_services = train["service"].value_counts().head(10)
print(top_services)
print()

subset = train[train["service"].isin(top_services.index)]
service_percent = pd.crosstab(subset["service"], subset["attack_class"], normalize="index") * 100
print(service_percent.round(1))

service_percent.plot(kind="bar", stacked=True, figsize=(10, 4))
plt.title("Class share inside the 10 most common services (training file)")
plt.ylabel("Percent of rows")
plt.show()

In [ ]:

cols_to_plot = ["duration", "src_bytes", "dst_bytes", "count"]

fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for i, c in enumerate(cols_to_plot):
    axes[0, i].hist(train[c], bins=50)
    axes[0, i].set_title(c + " (raw)")
    axes[1, i].hist(np.log1p(train[c]), bins=50)
    axes[1, i].set_title(c + " (log)")
plt.tight_layout()
plt.show()

# Correlation heatmap

In [ ]:
num_cols = train.select_dtypes(include="number").columns.drop("is_attack")
corr = train[num_cols].corr()

plt.figure(figsize=(14, 10))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Correlation between numeric columns (training file)")
plt.show()

# ---- 18b. Most correlated pairs ----
pairs = corr.abs().unstack()
pairs = pairs[pairs < 1].sort_values(ascending=False)


#Most correlated pairs

In [ ]:
pairs = corr.abs().unstack()
pairs = pairs[pairs < 1].sort_values(ascending=False)
print(pairs.head(20))

In [ ]:
print(corr.loc["root_shell", "su_attempted"])
print(corr.loc["dst_host_rerror_rate", "dst_host_srv_rerror_rate"])

# Normal vs Attack

In [ ]:
means = train.groupby("is_attack")[num_cols].mean().T
means.columns = ["Normal (0)", "Attack (1)"]
print(means.round(3))

# columns on a 0 to 1 scale

In [ ]:
# ---- 19b. Chart for columns on a 0 to 1 scale ----
chosen = ["logged_in", "serror_rate", "rerror_rate", "same_srv_rate", "dst_host_same_srv_rate"]
means.loc[chosen].plot(kind="bar", figsize=(9, 4))
plt.title("Average value: Normal vs Attack")
plt.ylabel("Average")
plt.show()

#Boxplots: Normal (0) vs Attack (1

In [ ]:
chosen = ["same_srv_rate", "serror_rate", "count", "dst_host_srv_count"]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, c in enumerate(chosen):
    sns.boxplot(data=train, x="is_attack", y=c, ax=axes[i])
    axes[i].set_title(c)
plt.tight_layout()
plt.show()

## 21. EDA Summary

**Data**
- Train: 125,973 rows, test: 22,544 rows, 41 features plus label (difficulty column dropped).
- Missing values and duplicates: (fill in from your cell 6 output).

**Classes**
- Train: Normal 67,343 / DOS 45,927 / PROBE 11,656 / R2L 995 / U2R 52.
- Test: Normal 9,711 / DOS 7,458 / PROBE 2,421 / R2L 2,754 / U2R 200.
- Strong imbalance: R2L is 0.8% of train but about 12% of test; U2R is 0.04% of train.
- 17 attack names appear only in the test file (all mapped to families).
- "Attack" is about 78% DOS, so Normal-vs-Attack averages mostly describe DOS.

**Text columns**
- protocol_type: icmp is mostly attacks, udp is mostly Normal, tcp is mixed.
- flag: S0 is 98.5% DOS, SF is 84.6% Normal; almost all R2L/U2R rows are in SF.
- service: 70 values; http, smtp, domain_u mostly Normal; ecr_i, eco_i, private mostly attacks.

**Numeric columns**
- num_outbound_cmds is 0 in every row (useless).
- src_bytes and dst_bytes are extremely skewed (maximum about 1.3 billion): need log or scaling.
- Redundant groups (correlation above 0.92): num_compromised/num_root; the four serror columns; the four rerror columns.
- Strong separators (boxplots): same_srv_rate, count, dst_host_srv_count; serror_rate is partial.

#Separate features (X) and targets (y

In [ ]:
drop_cols = ["label", "attack_class", "is_attack", "num_outbound_cmds"]

X_train_full = train.drop(columns=drop_cols)
X_test       = test.drop(columns=drop_cols)

y_train_multi = train["attack_class"]
y_test_multi  = test["attack_class"]
y_train_bin   = train["is_attack"]
y_test_bin    = test["is_attack"]

print(X_train_full.shape, X_test.shape)
print(list(X_train_full.columns) == list(X_test.columns))

# Split training file into training and validation parts
#

In [ ]:
from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr_multi, y_val_multi = train_test_split(
    X_train_full, y_train_multi,
    test_size=0.2, stratify=y_train_multi, random_state=42)

# same rows for the Normal-vs-Attack version
y_tr_bin  = y_train_bin.loc[y_tr_multi.index]
y_val_bin = y_train_bin.loc[y_val_multi.index]

print(X_tr.shape, X_val.shape)
print(y_val_multi.value_counts())

# Preprocessing: text to numbers, and scaling


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

text_features = ["protocol_type", "service", "flag"]
num_features  = [c for c in X_tr.columns if c not in text_features]

pre = ColumnTransformer([
    ("text", OneHotEncoder(handle_unknown="ignore"), text_features),
    ("num",  StandardScaler(), num_features)
])

# quick check: learn from training part only, then apply to validation
pre.fit(X_tr)
Z_tr  = pre.transform(X_tr)
Z_val = pre.transform(X_val)
print(len(num_features), "numeric features")
print(Z_tr.shape, Z_val.shape)

#Baseline model: Logistic Regression (Normal vs Attack)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

log_model = Pipeline([
    ("pre", pre),
    ("model", LogisticRegression(max_iter=1000))
])

log_model.fit(X_tr, y_tr_bin)                       # training

val_pred = log_model.predict(X_val)                 # predicting
print("Validation accuracy:", accuracy_score(y_val_bin, val_pred))
print(classification_report(y_val_bin, val_pred, target_names=["Normal", "Attack"]))

#Confusion matrix: Logistic Regression (Normal vs Attack)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

print(confusion_matrix(y_val_bin, val_pred))

ConfusionMatrixDisplay.from_predictions(
    y_val_bin, val_pred, display_labels=["Normal", "Attack"])
plt.title("Logistic Regression: validation")
plt.show()

#Decision Tree and Random Forest (Normal vs Attack)

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

tree_model = Pipeline([
    ("pre", pre),
    ("model", DecisionTreeClassifier(random_state=42))
])
tree_model.fit(X_tr, y_tr_bin)
tree_pred = tree_model.predict(X_val)
print("Decision Tree accuracy:", accuracy_score(y_val_bin, tree_pred))
print(confusion_matrix(y_val_bin, tree_pred))

forest_model = Pipeline([
    ("pre", pre),
    ("model", RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42))
])
forest_model.fit(X_tr, y_tr_bin)
forest_pred = forest_model.predict(X_val)
print("Random Forest accuracy:", accuracy_score(y_val_bin, forest_pred))
print(confusion_matrix(y_val_bin, forest_pred))

# Compare the three models (Normal vs Attack)

In [ ]:
results = []
for name, pred in [("Logistic Regression", val_pred),
                   ("Decision Tree", tree_pred),
                   ("Random Forest", forest_pred)]:
    tn, fp, fn, tp = confusion_matrix(y_val_bin, pred).ravel()
    results.append({"model": name,
                    "accuracy": round(accuracy_score(y_val_bin, pred), 4),
                    "missed_attacks": fn,
                    "false_alarms": fp})

comparison = pd.DataFrame(results)
print(comparison)

#Train the same three models on all 5 classes

In [ ]:
from sklearn.metrics import f1_score

models_multi = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42),
}

fitted_multi = {}
rows = []
for name, clf in models_multi.items():
    pipe = Pipeline([("pre", pre), ("model", clf)])
    pipe.fit(X_tr, y_tr_multi)                      # training
    pred = pipe.predict(X_val)                      # predicting
    fitted_multi[name] = pipe
    rows.append({"model": name,
                 "accuracy": round(accuracy_score(y_val_multi, pred), 4),
                 "macro_f1": round(f1_score(y_val_multi, pred, average="macro"), 4)})

print(pd.DataFrame(rows))

#Per-class scores: Random Forest (5 classes)

In [ ]:
rf_multi = fitted_multi["Random Forest"]
rf_pred = rf_multi.predict(X_val)

print(classification_report(y_val_multi, rf_pred))

order = ["Normal", "DOS", "PROBE", "R2L", "U2R"]
cm = confusion_matrix(y_val_multi, rf_pred, labels=order)
print(pd.DataFrame(cm, index=order, columns=order))

#Random Forest with class weights (5 classes)

In [ ]:
rf_weighted = Pipeline([
    ("pre", pre),
    ("model", RandomForestClassifier(n_estimators=100, n_jobs=-1,
                                     random_state=42, class_weight="balanced"))
])
rf_weighted.fit(X_tr, y_tr_multi)
w_pred = rf_weighted.predict(X_val)

print("Macro F1 (before):  ", round(f1_score(y_val_multi, rf_pred, average="macro"), 4))
print("Macro F1 (weighted):", round(f1_score(y_val_multi, w_pred, average="macro"), 4))
print(classification_report(y_val_multi, w_pred))

#Random Forest on the official test file

In [ ]:
final_model = fitted_multi["Random Forest"]
test_pred = final_model.predict(X_test)

print("Test accuracy:", round(accuracy_score(y_test_multi, test_pred), 4))
print("Test macro F1:", round(f1_score(y_test_multi, test_pred, average="macro"), 4))
print(classification_report(y_test_multi, test_pred))

order = ["Normal", "DOS", "PROBE", "R2L", "U2R"]
cm = confusion_matrix(y_test_multi, test_pred, labels=order)
print(pd.DataFrame(cm, index=order, columns=order))

#Which attack names are missed

In [ ]:
check = test[["label", "attack_class"]].copy()
check["predicted"] = test_pred
check["correct"] = (check["predicted"] == check["attack_class"])
check["seen_in_training"] = check["label"].isin(train["label"])

by_name = check.groupby("label").agg(
    rows=("correct", "size"),
    correct_rate=("correct", "mean"),
    seen_in_training=("seen_in_training", "first")
).round(2)
print(by_name.sort_values("rows", ascending=False))

attacks_only = check[check["attack_class"] != "Normal"]
print(attacks_only.groupby("seen_in_training")["correct"].agg(["size", "mean"]))

#Save the model

In [ ]:
import pickle, sklearn

with open("model.pkl", "wb") as f:
    pickle.dump(final_model, f)

with open("columns.pkl", "wb") as f:
    pickle.dump(list(X_train_full.columns), f)

print("model.pkl size (MB):", round(os.path.getsize("model.pkl") / (1024 * 1024), 1))
print("scikit-learn version:", sklearn.__version__)
print("pandas version:", pd.__version__)


#Check the saved file works

In [ ]:
with open("model.pkl", "rb") as f:
    loaded = pickle.load(f)

check_pred = loaded.predict(X_test.head(1000))
print("Same answers as before:", (check_pred == test_pred[:1000]).all())

#One row, the way the app will send it

In [ ]:
import sys, json


one_row = X_test.iloc[[0]]                       # double brackets keep it as a table
print("True class:", test["attack_class"].iloc[0])
print("Model says:", loaded.predict(one_row)[0])
class_names = loaded.named_steps["model"].classes_
print(dict(zip(class_names, loaded.predict_proba(one_row)[0].round(3))))

#Allowed values for the dropdown menus

In [ ]:
options = {c: sorted(train[c].unique().tolist())
           for c in ["protocol_type", "service", "flag"]}
with open("options.json", "w") as f:
    json.dump(options, f)

# Two example rows per class, taken from the test file


In [ ]:
cols = list(X_train_full.columns)
parts = []
for cls in ["Normal", "DOS", "PROBE", "R2L", "U2R"]:
    parts.append(test[test["attack_class"] == cls].sample(2, random_state=42))
examples = pd.concat(parts)
examples_out = examples[cols].copy()
examples_out["true_class"] = examples["attack_class"]
examples_out.to_csv("examples.csv", index=False)
print(pd.DataFrame({"true_class": examples["attack_class"],
                    "model_says": loaded.predict(examples[cols])}))

In [ ]:
import numpy as np
print("Python:", sys.version.split()[0], "| numpy:", np.__version__)

# Download the app files

In [ ]:

import zipfile
from google.colab import files

app_files = ["model.pkl", "columns.pkl", "options.json", "examples.csv"]

for name in app_files:
    print(name, "->", round(os.path.getsize(name) / (1024 * 1024), 2), "MB")

with zipfile.ZipFile("ThreadLens_files.zip", "w") as z:
    for name in app_files:
        z.write(name)

files.download("ThreadLens_files.zip")